# Модуль 1. Вход в стенд и базовая диагностика

Этот ноутбук является канонической практикой Модуля 1.

На выходе ты должен:

- понимать, что стенд уже поднят на хосте;
- подключиться к Spark-кластеру из ноутбука;
- выполнить базовый smoke test;
- сопоставить сервисы стенда с их ролями;
- знать первый маршрут диагностики через `docker compose ps -a`, логи и UI.


## Перед стартом

Если стенд еще не готов, выполни подготовку и запуск по [START_HERE](../START_HERE.md#быстрый-запуск).
Если возвращаешься после паузы, используй [памятку учащегося](../docs/study_workflow.md).

Важно: `docker compose build`, `docker compose up -d`, `docker compose ps -a` и просмотр логов выполняются в терминале хоста; терминал браузерного VS Code работает внутри контейнера.

Ожидаемое состояние перед началом:

- контейнеры `spark-master`, `spark-worker-1`, `spark-worker-2`, `minio`, `postgres-iceberg`, `trino`, `jupyter`, `ide` уже подняты;
- открываются `Spark UI`, `Trino UI`, `MinIO Console` и браузерный VS Code;
- ты выбрал ядро Python `/usr/bin/python3` внутри контейнера `ide` (Select Kernel → Python Environments).


## Карта сервисов

| Сервис | Роль |
| --- | --- |
| `MinIO` | `storage`: объектное хранилище для данных и файлов Iceberg |
| `PostgreSQL` | `catalog`: хранит метаданные JDBC-каталога Iceberg |
| `Spark` | `compute`: выполняет PySpark и Spark SQL |
| `Trino` | `compute`: читает те же таблицы через общий каталог |
| `VS Code / JupyterLab` | Точка входа в учебные ноутбуки |

В этом модуле мы не строим пайплайн и не создаём Iceberg-таблицы. Задача только одна: убедиться, что среда рабочая и понятная.


In [ ]:
from spark.cluster_smoke import (
    create_spark_session,
    format_smoke_report,
    run_cluster_smoke,
)

spark = create_spark_session(app_name="module-01-environment-and-smoke-test")
spark


## Демонстрация: подключение к кластеру

Сначала посмотрим на базовые признаки того, что ноутбук говорит именно с кластером, а не с локальным `local[*]` режимом.


In [ ]:
connection_details = {
    "app_name": spark.sparkContext.appName,
    "master_url": spark.sparkContext.master,
    "default_parallelism": spark.sparkContext.defaultParallelism,
    "spark_version": spark.version,
}
connection_details


Проверь глазами:

- `master_url` должен быть `spark://spark-master:7077`, а не `local[*]`;
- `default_parallelism` должен быть больше `1`;
- в `Spark UI` должны быть видны master и worker-ы.


## Демонстрация: Spark smoke test

Smoke test запускает распределённое вычисление на `range(0, 1_000_000)` и сверяет детерминированный результат.

Если тест проходит, это означает минимум следующее:

- ядро ноутбука может создать SparkSession;
- Spark подключён к кластерному master;
- задание действительно выполняется, а не падает на первом действии.


In [ ]:
smoke_report = run_cluster_smoke(spark)
print(format_smoke_report(smoke_report))
smoke_report


## Самостоятельное повторение

Сделай руками и проверь себя:

1. Открой `Spark UI`, `Trino UI`, `MinIO Console` и браузерный VS Code.
2. Сопоставь каждый сервис с одной из ролей: `storage`, `catalog`, `compute`, `entrypoint`.
3. Объясни, почему этот smoke test не должен работать в `local[*]` режиме.
4. На хосте выполни `docker compose ps -a` и посмотри, какие контейнеры должны быть в состоянии `Up`.
5. На хосте выполни `docker compose logs -f trino` или `docker compose logs -f spark-master`, чтобы посмотреть, где начинается первичная диагностика.


In [ ]:
service_roles = {
    # Заполни значения самостоятельно: storage, catalog, compute или entrypoint.
    "MinIO": "TODO",
    "PostgreSQL": "TODO",
    "Spark": "TODO",
    "Trino": "TODO",
    "IDE": "TODO",
}
service_roles


## Checkpoint

К концу модуля ты должен уметь подтвердить:

- стенд поднят и основные контейнеры живы;
- `Spark UI`, `MinIO Console`, `Trino UI` и браузерный VS Code открываются;
- smoke test напечатал `Spark cluster smoke test passed.` и `result_sum=499999500000`;
- ты понимаешь назначение `MinIO`, `PostgreSQL`, `Spark`, `Trino` и IDE;
- ты знаешь, что первый маршрут диагностики начинается с `docker compose ps -a`, `docker compose logs -f <service>` и проверки UI.

После последней ячейки `spark.stop()` сохрани ноутбук. Дальше — [Модуль 2](./02_lakehouse_mental_model.ipynb)
или [завершение занятия](../docs/study_workflow.md#закончить-занятие).
Все уроки собраны в [маршруте по модулям](../docs/course_program.md#маршрут-по-модулям).

In [ ]:
spark.stop()
